# Subsequent Search Misses and Attentional Blinks
temporal effects on LWS probability, with reference to (1) the start of the trial; (2) the most recent target detection - while taking into account the "type" of the target

In [1]:
import os
from enum import Enum, auto

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

import config as cnfg
from analysis.helpers.read_data import read_data
from analysis.helpers.funnels.build_funnels import build_event_classification_funnel

pio.renderers.default = "notebook"      # "notebook" or "browser"

## Prepare Data

In [2]:
funnel_results = build_event_classification_funnel(
    cnfg.OUTPUT_PATH, "lws", "visit", exclude="invalid_trials"
)

# drop irrelevant columns
funnel_results = funnel_results.drop(columns=[
    "upto_gaze_coverage", "upto_fixation_rate", "upto_no_bad_action", "upto_no_miss_with_false_alarm",
    "upto_before_identification", "upto_after_identification", "upto_not_close_to_trial_end", "upto_not_before_exemplar_visit"
], errors="ignore")

In [3]:
loaded_data = read_data(cnfg.OUTPUT_PATH)
metadata = loaded_data.metadata
fixations = loaded_data.fixations
idents = loaded_data.identifications
targets = loaded_data.targets

hits = (
    idents
    .loc[idents["identification_category"] == "hit"]
    .drop(columns=[
        "identification_category", 'left_x', 'left_y', 'left_pupil', 'right_x', 'right_y', 'right_pupil'
    ])
    .merge(
        targets[["subject", "trial", "target", "category"]], on=["subject", "trial", "target"], how="left"
    )
    .rename(columns={"category": "target_category"})
    .sort_values(["subject", "trial", "time"])
    .reset_index(drop=True)
)
del loaded_data, idents

add `num_targets` to the funnel results

In [4]:
funnel_results = funnel_results.merge(
    metadata[["subject", "trial", "num_targets"]], on=["subject", "trial"], how="left"
)
del metadata

add previously-identified target information to the funnel results

In [5]:
def get_target_history(event_row: pd.Series):
    # filter hits for this subject&trial that happened before this event started
    prior_hits = hits[
        (hits['subject'] == event_row['subject']) &
        (hits['trial'] == event_row['trial']) &
        (hits['time'] < event_row['start_time'])
    ]
    if prior_hits.empty:
        time_since_id = np.nan
    else:
        most_recent_time = prior_hits['time'].max()
        time_since_id = event_row['start_time'] - most_recent_time
    return pd.Series({
        'num_targets_found_before': len(prior_hits),
        'target_categories_found_before': prior_hits['target_category'].tolist(),
        'time_since_recent_find': time_since_id,
    })


# append identified target counts and their categories to funnel dataframe
funnel_results = pd.concat([funnel_results, funnel_results.apply(get_target_history, axis=1)], axis=1)

add fixation counts from trial start and from recent target identification to the funnel results

In [6]:
def count_fixations_from_trial_onset(event_row: pd.Series):
    relevant_fixations = fixations.loc[
        (fixations['subject'] == event_row['subject']) &
        (fixations['trial'] == event_row['trial']) &
        (fixations['eye'] == event_row['eye']) &
        (fixations['end_time'] < event_row['start_time'])
    ]
    return len(relevant_fixations)

def count_fixations_since_last_hit(event_row: pd.Series):
    # find the most recent hit before the current visit
    prior_hits = hits.loc[
        (hits['subject'] == event_row['subject']) &
        (hits['trial'] == event_row['trial']) &
        (hits['time'] < event_row['start_time'])
    ]
    if prior_hits.empty:    # no prior hit, so 0 fixations since "last" hit
        return 0
    last_hit_time = prior_hits['time'].max()

    # find fixations after the hit & before the current visit
    inter_fixations = fixations.loc[
        (fixations['subject'] == event_row['subject']) &
        (fixations['trial'] == event_row['trial']) &
        (fixations['eye'] == event_row['eye']) &
        (fixations['start_time'] > last_hit_time) &
        (fixations['end_time'] < event_row['start_time'])
    ]
    return len(inter_fixations)


# append fixation counts to the funnel dataframe
funnel_results['fixations_since_trial_start'] = funnel_results.apply(count_fixations_from_trial_onset, axis=1)
funnel_results['fixations_since_last_hit'] = funnel_results.apply(count_fixations_since_last_hit, axis=1)

### Classify Visits
We can consider LWS sub-types based on the number and type of targets found so far in the trial, as well as non-LWS visits - e.g., the fixation/event co-occuring with target identification or returning to a previously identified target.

In [7]:
class VisitTypeEnum(Enum):
    """ An enum classification of target-visits based on the types of targets that were identified prior to the visit """
    BEFORE_HIT = auto()             # no targets found yet
    IDENTIFICATION_VISIT = auto()   # the event coinciding with target identification
    TARGET_RETURN = auto()          # returning to a previously identified target
    AFTER_1HIT_SAME = auto()        # 1 found; same category as current
    AFTER_1HIT_DIFF = auto()        # 1 found; different category as current
    AFTER_2HIT_SAME_DIFF = auto()   # 2 found; first was same category as current, then a different category
    AFTER_2HIT_DIFF_SAME = auto()   # 2 found; first was a different category from current, then a same category
    AFTER_2HIT_DIFF_DIFF = auto()   # 2 found; both were from a different category than current
    OTHER = auto()                  # fall-back


def classify_event(event_row: pd.Series) -> VisitTypeEnum:
    # check if *any* target was identified in the trial
    trial_hits = hits.loc[
        (hits['subject'] == event_row['subject']) & (hits['trial'] == event_row['trial'])
    ].sort_values('time')
    if trial_hits.empty:
        return VisitTypeEnum.BEFORE_HIT
    if all(trial_hits['time'] > event_row['end_time']):
        return VisitTypeEnum.BEFORE_HIT
    # check if *this* target was identified before/during this visit
    this_target_hit = trial_hits.loc[trial_hits["target"] == event_row['target']]
    if not this_target_hit.empty:
        this_target_hit_time = this_target_hit['time'].iloc[0]
        if this_target_hit_time < event_row['start_time']:
            return VisitTypeEnum.TARGET_RETURN
        if event_row['start_time'] <= this_target_hit_time <= event_row['end_time']:
            return VisitTypeEnum.IDENTIFICATION_VISIT
    # check which *other* targets were identified prior to this visit

    return None


class VisitType(Enum):
    # --- LWS Subtypes (Misses) ---
    LWS_BEFORE_ANY_HIT = auto()         # No targets found yet
    LWS_AFTER_HIT_SAME = auto()         # 1 found; same category as current
    LWS_AFTER_HIT_DIFF = auto()         # 1 found; different category
    LWS_AFTER_2HIT_MIXED = auto()       # 2 found; one same, one different
    LWS_AFTER_2HIT_BOTH_DIFF = auto()   # 2 found; both different from current

    # --- Non-LWS Target Visits ---
    IDENTIFICATION_VISIT = auto()       # The moment of discovery/hit
    TARGET_RETURN = auto()              # Re-visiting a target already identified

    # --- Miscellaneous ---
    OTHER = auto()                      # Fallback (e.g., target visit succeeded by a bottom-strip visit


def classify_visit(row) -> VisitType:
    trial_hits = hits.loc[
        (hits['subject'] == row['subject']) & (hits['trial'] == row['trial'])
    ].sort_values('time')
    target_hits = trial_hits.loc[trial_hits["target"] == row['target']]
    assert len(target_hits) <= 1, f"A single target has multiple hits! subject: {row['subject']}, trial: {row['trial']}"

    if len(target_hits) == 1:
        hit_time = target_hits['time'].iloc[0]
        if hit_time < row['start_time']:
            return VisitType.TARGET_RETURN      # returning to a previously identified target
        if row['start_time'] <= hit_time and hit_time <= row['end_time']:
            return VisitType.IDENTIFICATION_VISIT   # identification event
    if row['is_lws']:
        n_found = row['num_targets_found_before']
        if n_found == 0:
            return VisitType.LWS_BEFORE_ANY_HIT
        same_cat_found = row['target_category'] in row['target_categories_found_before']
        if n_found == 1:
            return VisitType.LWS_AFTER_HIT_SAME if same_cat_found else VisitType.LWS_AFTER_HIT_DIFF
        if n_found == 2:
            return VisitType.LWS_AFTER_2HIT_MIXED if same_cat_found else VisitType.LWS_AFTER_2HIT_BOTH_DIFF
    return VisitType.OTHER


# append visit classifications to the funnel dataframe
funnel_results['visit_type'] = funnel_results.apply(classify_visit, axis=1)

In [8]:
funnel_results["visit_type"].value_counts(dropna=False)

visit_type
VisitType.IDENTIFICATION_VISIT        1007
VisitType.TARGET_RETURN                529
VisitType.LWS_BEFORE_ANY_HIT           342
VisitType.LWS_AFTER_HIT_DIFF           225
VisitType.OTHER                        191
VisitType.LWS_AFTER_2HIT_MIXED          25
VisitType.LWS_AFTER_HIT_SAME            16
VisitType.LWS_AFTER_2HIT_BOTH_DIFF       6
Name: count, dtype: int64